# 使い方

YAIBA-VRCで取得したログデータを分析し、ヒストグラム・ヒートマップ・統計データ・動画を生成し、ZIPファイルで出力します。

## 手順

1. プライバシーポリシーと同意事項を読む
2. ライブラリのインストール
3. ファイル選択（`./data/logfiles/` 配下のログファイルを使用）
4. 処理開始

## プライバシーポリシーと同意事項

【重要】 プライバシーとデータの取り扱いについて
- 本ツールは、VRChatワールドギミック「YAIBA-VRC」によって位置情報を回転情報の取得に**同意したユーザーのログ**のみを扱うことを想定しています。
- ファイルをアップロードすることにより、データがGoogle Colaboratoryのサーバーに送信され、処理されることに同意したものとみなします。
  これは**第三者提供**に準ずる行為となりますので、取り扱うデータの内容には十分ご注意ください。

In [ ]:
# @title ライブラリのインストール

# フォントファイルのインストール
!apt-get -qq -y update
!apt-get -qq -y install fonts-noto-cjk fonts-ipafont-gothic

# YAIBA
!pip install --quiet git+https://github.com/ScienceAssembly/YAIBA.git

# YAIBA-BI
!pip install --quiet git+https://github.com/earl-klutz/yaiba-bi.git

In [ ]:
# @title ファイル選択

import os
import glob
import shutil
import ipywidgets as widgets
from IPython.display import display
from yaiba_bi import core


def get_display_box(e: Exception) -> widgets.HTML:
    contents = widgets.HTML(
        value=f"<div style='padding:10px; border:2px solid orange; background:#FFF3E0; color:#E65100;'>"
              f"<b>エラー:</b> {str(e)}</div>"
    )
    return contents


# ディレクトリ設定
DATE_STR = "20260514"  # データの日付（例: "20260430"）
INPUT_DIR = f"./data/logfiles/{DATE_STR}"
OUTPUT_DIR = f"./data/output/{DATE_STR}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ログファイル検索（日時順）
log_files = sorted(glob.glob(os.path.join(INPUT_DIR, "output_log_*.txt")))

if len(log_files) == 0:
    exception = FileNotFoundError(f"{INPUT_DIR} にログファイルが見つかりません")
    display(get_display_box(exception))
    raise exception

print("利用可能なログファイル:")
for i, f in enumerate(log_files):
    print(f"  [{i}] {os.path.basename(f)}")

# 使用するファイルのインデックス（-1 = 最新）
# 特定のファイルを使う場合はインデックスを変更してください
SELECTED_INDEX = -1

path = log_files[SELECTED_INDEX]
print(f"\n使用するファイル: {os.path.basename(path)}")

# LogDataオブジェクト作成
try:
    log_data: core.LogData = core.load(path)
except ValueError as e:
    display(get_display_box(e))
    raise e

In [ ]:
# @title 処理開始

# 動画作成処理を実行するかを決めるBool値
# @markdown 動画を作成する場合はチェックをつけてください
is_generate_movie = False  # @param {"type": "boolean"}

if os.path.isfile("./output.zip"):
    os.remove("./output.zip")

# 処理
try:
    position = log_data.get_position()
    attendance = log_data.get_attendance()
    area = log_data.get_area()

    heatmap_generator = core.HeatmapGenerator(area)
    heatmap_generator.run(position.copy(), "heatmap", OUTPUT_DIR)

    io_params = core.HistIOParams(out_dir=OUTPUT_DIR)
    core.run_histogram_mvp(df=position.copy(), io=io_params, output_basename="histogram")

    if is_generate_movie:
        movie_io_params = core.MovieIOParams(out_dir=OUTPUT_DIR)
        core.run_movie_xz(log_data.get_position(), io=movie_io_params)

    event_date = log_data.get_position()["event_day"].unique()[0]
    render_config = core.RenderConfig(str(event_date), "sample")
    trajectory_config = core.TrajectoryConfig()

    visualizer = core.EventLogVisualizer(render_config, trajectory_config)
    visualizer.run(log_data.get_attendance(), log_data.get_position(), log_data.get_area())
except Exception as e:
    display(get_display_box(e))
    raise e

# ZIPファイル作成
shutil.make_archive("output", format="zip", root_dir=OUTPUT_DIR)
print("output.zip を生成しました")